# `popcorn.compile` (experimental)

Inject popcorn into a model that never imports it: `enable()` installs an inductor pass that rewrites matched subgraphs to dispatched kernels. Here we leave [`modeling_butter.py`](./modeling_butter.py) on its plain-torch `OPS` and only rewrite the fused loss (fp32; fused LCE implementations are float32-only).

In [ ]:
import torch

from modeling_butter import ButterConfig, ButterForCausalLM
from popcorn.bench import compare, report
import popcorn.compile

torch.manual_seed(0)
device, dtype = "cuda", torch.float32
config = ButterConfig(layer_types=("attn",), vocab_size=131072)
model = ButterForCausalLM(config).to(device, dtype)
batch = {
    "input_ids": torch.randint(config.vocab_size, (16, 1024), device=device),
    "labels": torch.randint(config.vocab_size, (16, 1024), device=device),
}


def train_step(**batch):
    loss = model(**batch)
    loss.backward()
    model.zero_grad(set_to_none=True)
    return loss.detach()


compiled = torch.compile(train_step)
plain = compare(compiled, train_step, batch, backward=False, repeats=3)
report(plain, "torch.compile", mine="compiled", reference="eager")

n = popcorn.compile.enable(ops=["linear_cross_entropy"])
torch._dynamo.reset()
print(f"{n} patterns registered")

with_popcorn = compare(compiled, train_step, batch, backward=False, repeats=3)
report(with_popcorn, "+ popcorn", mine="compiled", reference="eager")
print(
    f"vs plain compile: {plain.bench['fwd_ms'] / with_popcorn.bench['fwd_ms']:.2f}x time, "
    f"{plain.bench['fwd_mem_mb'] / with_popcorn.bench['fwd_mem_mb']:.1f}x memory"
)